# NB06 — Code Quality: ruff, black, mypy, pre-commit

**Module 16 — Research Software Engineering**

---

## Motivation

Code that is syntactically correct can still be wrong in subtle ways: a mutable default argument silently shares state between calls; a missing type annotation lets an integer flow into a function expecting a string; a line that's 150 characters wide is unreadable on a standard display. Code quality tools catch these problems automatically, before tests, before code review, and before the bug reaches a user.

## Intuition

- **ruff**: an extremely fast Python linter and formatter (written in Rust). It replaces flake8, isort, pyupgrade, and parts of pylint in a single tool.
- **black**: an opinionated formatter that enforces a consistent style. "Black is the formatting" — there are no options to argue about.
- **mypy**: a static type checker. It reads type annotations and reports mismatches before the code is run.
- **pre-commit**: a framework that runs configurable hooks before each `git commit`. If any hook fails, the commit is blocked.

## Biological Background

None of these tools are biology-specific, but they are used by every major scientific Python project: NumPy, SciPy, scikit-learn, Biopython, scanpy. When you contribute to these projects, your PR will be rejected if it doesn't pass their linters and type checks. More importantly, type annotations on bioinformatics functions prevent entire classes of bugs: passing a protein sequence where a DNA sequence is expected, passing counts where fractions are expected.

---

In [ ]:
import ast
import textwrap
from dataclasses import dataclass, field
from typing import Any

# Demonstrate: a function with multiple code quality problems
BAD_CODE = textwrap.dedent("""
def gc(s, result=[]):  # mutable default argument!
    GC = 0
    for b in s:
        if b=='G' or b=='C':
            GC+=1
    result.append(GC/len(s))
    return result
""")

GOOD_CODE = textwrap.dedent("""
def gc_content(seq: str) -> float:
    """Compute GC content as a fraction in [0.0, 1.0]."""
    if not seq:
        return 0.0
    seq = seq.upper()
    return (seq.count('G') + seq.count('C')) / len(seq)
""")

print('BAD CODE (before refactoring):')
print(BAD_CODE)
print('GOOD CODE (after refactoring):')
print(GOOD_CODE)

In [ ]:
# Implement a simple linting audit using the AST
# This replicates some of what ruff and mypy do, at a simplified level

@dataclass
class LintIssue:
    line: int
    code: str
    message: str


def lint_audit(source: str) -> list[LintIssue]:
    """Run a simplified lint audit on Python source code.

    Checks for:
    - Missing return type annotation (MTA001)
    - Missing docstring (MDA001)
    - Lines > 88 characters (E501)
    - Mutable default arguments (B006)

    Parameters
    ----------
    source : str
        Python source code as a string.

    Returns
    -------
    list[LintIssue]
        List of lint issues found, in line order.
    """
    issues: list[LintIssue] = []

    try:
        tree = ast.parse(source)
    except SyntaxError as e:
        issues.append(LintIssue(e.lineno or 0, 'E999', f'SyntaxError: {e}'))
        return issues

    # Check lines
    for i, line in enumerate(source.splitlines(), start=1):
        if len(line) > 88:
            issues.append(LintIssue(i, 'E501', f'Line too long ({len(line)} > 88 chars)'))

    # Check function definitions
    for node in ast.walk(tree):
        if not isinstance(node, ast.FunctionDef):
            continue

        # Missing return type annotation
        if node.returns is None:
            issues.append(LintIssue(node.lineno, 'MTA001',
                                     f'Function {node.name!r} missing return type annotation'))

        # Missing docstring
        has_docstring = (
            node.body
            and isinstance(node.body[0], ast.Expr)
            and isinstance(node.body[0].value, ast.Constant)
            and isinstance(node.body[0].value.value, str)
        )
        if not has_docstring:
            issues.append(LintIssue(node.lineno, 'MDA001',
                                     f'Function {node.name!r} missing docstring'))

        # Mutable default arguments
        for default in node.args.defaults:
            if isinstance(default, (ast.List, ast.Dict, ast.Set)):
                issues.append(LintIssue(node.lineno, 'B006',
                                         f'Function {node.name!r} uses mutable default argument'))

    return sorted(issues, key=lambda x: x.line)


# Test on bad and good code
for label, code in [('BAD CODE', BAD_CODE), ('GOOD CODE', GOOD_CODE)]:
    issues = lint_audit(code)
    print(f'\n{label}: {len(issues)} issue(s)')
    for issue in issues:
        print(f'  Line {issue.line}: [{issue.code}] {issue.message}')

In [ ]:
# Type annotations: what mypy checks
typed_examples = [
    ('def f(x: str) -> float:\n    return len(x)', 'Correct: str → float'),
    ('def f(x: str) -> float:\n    return x',       'Error: returning str, expected float'),
    ('def f(x) -> float:\n    return len(x)',        'Missing param annotation'),
]

print('Type annotation examples for mypy:')
for code, label in typed_examples:
    print(f'\n  [{label}]')
    print(textwrap.indent(code, '    '))

# NumPy array type annotations
np_annotation_example = textwrap.dedent("""
    import numpy as np
    from numpy.typing import NDArray

    def normalize(arr: NDArray[np.float64]) -> NDArray[np.float64]:
        \"\"\"Normalize an array to sum to 1.\"\"\"
        total = arr.sum()
        if total == 0:
            raise ValueError('Cannot normalize a zero array')
        return arr / total
""")

print('\nNumPy array type annotation:')
print(np_annotation_example)

In [ ]:
# Generate .pre-commit-config.yaml
import yaml
from pathlib import Path
import tempfile

pre_commit_config = {
    'repos': [
        {
            'repo': 'https://github.com/astral-sh/ruff-pre-commit',
            'rev': 'v0.4.4',
            'hooks': [
                {'id': 'ruff', 'args': ['--fix']},
                {'id': 'ruff-format'},
            ],
        },
        {
            'repo': 'https://github.com/kynan/nbstripout',
            'rev': '0.7.1',
            'hooks': [{'id': 'nbstripout'}],
        },
        {
            'repo': 'https://github.com/pre-commit/mirrors-mypy',
            'rev': 'v1.10.0',
            'hooks': [
                {
                    'id': 'mypy',
                    'args': ['--strict'],
                    'additional_dependencies': ['numpy'],
                }
            ],
        },
    ]
}

config_yaml = yaml.dump(pre_commit_config, default_flow_style=False)
print('.pre-commit-config.yaml:')
print(config_yaml)

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Code Quality Tools', fontsize=14, fontweight='bold')

# --- Panel 1: Code quality score before/after ---
ax1 = axes[0]
ax1.set_title('Code Quality: Before vs After Refactoring', fontweight='bold')

categories = ['Return type\nannotation', 'Docstring', 'No mutable\ndefaults',
              'Line length\n<= 88', 'Consistent\nformatting']
before = [0, 0, 0, 0.5, 0.3]
after  = [1, 1, 1, 1.0, 1.0]

x = np.arange(len(categories))
width = 0.35

ax1.bar(x - width/2, before, width, label='Before (bad_code)', color='#e74c3c', alpha=0.8)
ax1.bar(x + width/2, after, width, label='After (good_code)', color='#27ae60', alpha=0.8)

ax1.set_xticks(x)
ax1.set_xticklabels(categories, fontsize=8)
ax1.set_ylabel('Score (0 = fail, 1 = pass)')
ax1.set_ylim(0, 1.3)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# --- Panel 2: Type coverage chart ---
ax2 = axes[1]
ax2.set_title('Type Annotation Coverage', fontweight='bold')

functions = ['gc_content', 'reverse_complement', 'melting_temperature',
             'bootstrap_ci', 'read_fasta', 'bad_function']
has_param_annot  = [1, 1, 1, 0.5, 0, 0]
has_return_annot = [1, 1, 1, 1,   0, 0]

x2 = np.arange(len(functions))
ax2.bar(x2 - 0.2, has_param_annot, 0.4, label='Param annotations', color='#3498db', alpha=0.8)
ax2.bar(x2 + 0.2, has_return_annot, 0.4, label='Return annotation', color='#9b59b6', alpha=0.8)

ax2.set_xticks(x2)
ax2.set_xticklabels(functions, rotation=25, ha='right', fontsize=8)
ax2.set_ylabel('Present (1) / Absent (0)')
ax2.set_ylim(0, 1.4)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Add a dashed line at 1.0
ax2.axhline(y=1.0, color='#27ae60', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('code_quality_overview.png', dpi=120, bbox_inches='tight')
plt.show()

## Exercises

See `exercises/README.md` → E06.

1. Take the `BAD_CODE` above and refactor it until `lint_audit` returns zero issues.
2. Add full type annotations to the refactored function. Check that mypy would accept it.
3. What is a mutable default argument? Write a test that demonstrates the bug it causes.

## Quiz

1. What does ruff replace (which older tools)?
2. What is the maximum line length black enforces by default?
3. What does `mypy --strict` enable that basic mypy does not?
4. What does pre-commit do when a hook fails?
5. What is the difference between `NDArray[np.float64]` and `np.ndarray` in type annotations?

## Reflection

*After completing:* Why do scientific programmers often skip type annotations? What specific bug have you seen or can you imagine that type annotations would have prevented?

## References

- ruff: https://docs.astral.sh/ruff/
- mypy: https://mypy.readthedocs.io/
- pre-commit: https://pre-commit.com/